In [1]:
USER_INFO_PATH = r'..\data\fe\user_info.parquet'

TOPIC_COMMENT_PATH = r"..\data\cleaned\topic_comment.parquet"
USER_WEIBO_PATH = r"..\data\cleaned\user_weibo.parquet"

In [2]:
import pandas as pd

df_user_info = pd.read_parquet(USER_INFO_PATH)

df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)
df_user_weibo = pd.read_parquet(USER_WEIBO_PATH)


In [ ]:
# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "user_info": len(df_user_info),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  user_info:     {len(df_user_info):>10,}")

In [ ]:
import re

def clean_text(text: str, type: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 移除 回复@xxx: 前缀（保留正文部分）
    4. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 移除 "回复@xxx：" 或 "回复@xxx:" 前缀，保留后续正文
    if type == "comment":
        text = re.sub(r'^回复@[\w\u4e00-\u9fff]+[：:]', '', text)

    # 4. 话题标签：#xxx# → xxx（保留标签文字内容）
    # text = re.sub(r'#([^#]+)#', r'\1', text)

    # 5. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text



In [ ]:
# ========== 5.4 description 清洗 ==========
df_user_info["description"] = df_user_info["description"].apply(
    lambda x: clean_text(x, type="description") if isinstance(x, str) else x
).replace('', None)

# ========== 5.5 粉丝/关注比（影响力指标）==========
df_user_info["follower_following_ratio"] = (
    df_user_info["follower_count"] / df_user_info["following_count"].replace(0, 1)
).round(2)
print(f"✅ 粉丝关注比计算完成")

In [ ]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# 保存
datasets = {
    "user_info": df_user_info,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")